# TIFF + filtered Spot Z viewer

確定済みの multi-page TIFF と `<spots>` XML を重ね、Z断面を移動しながら確認するviewerです。

- Dataset: 042926 / 050126 / 062226
- Z slider / Play: 断面移動と連続再生
- Zoom / Center X / Center Y: 表示領域の拡大と移動
- Contrast percentiles: 元画像の表示コントラスト
- Marker size / color: Spot表示の調整

画像は必要なZだけ遅延読み込みし、直近12枚をメモリに保持します。XMLはDatasetを初めて選択した時だけ読み込みます。

In [7]:
from collections import OrderedDict
from functools import lru_cache
from pathlib import Path
import xml.etree.ElementTree as ET

import ipywidgets as widgets
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
import numpy as np
import tifffile as tiff

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Project root containing src/ was not found.')

DATASETS = OrderedDict([
    ('042926', {
        'tif': PROJECT_ROOT / 'data' / '042926_MAY08R_FOS_1_retake_c.tif',
        'processed_dir': PROJECT_ROOT / 'outputs' / '042926_MAY08R_FOS_1_retake_c_uint16_scale10000_angle_smoothed',
        'xml': PROJECT_ROOT / 'outputs' / 'trackmate_042926_angle_smoothed_final' / '042926_MAY08R_FOS_1_retake_c_filtered_spots.xml',
    }),
    ('050126', {
        'tif': PROJECT_ROOT / 'data' / '050126_MAY08R_FOS_2_re_c.tif',
        'processed_dir': PROJECT_ROOT / 'outputs' / '050126_MAY08R_FOS_2_re_c_uint16_scale10000_angle_smoothed',
        'xml': PROJECT_ROOT / 'outputs' / 'trackmate_050126_angle_smoothed_final' / '050126_MAY08R_FOS_2_re_c_filtered_spots.xml',
    }),
    ('062226', {
        'tif': PROJECT_ROOT / 'data' / '062226_MAY05R_FOS_1_re2_c.tif',
        'processed_dir': PROJECT_ROOT / 'outputs' / '062226_MAY05R_FOS_1_re2_c_uint16_scale10000_angle_smoothed',
        'xml': PROJECT_ROOT / 'outputs' / 'trackmate_062226_angle_smoothed_final' / '062226_MAY05R_FOS_1_re2_c_filtered_spots.xml',
    }),
])

for dataset_name, paths in DATASETS.items():
    for kind, path in paths.items():
        exists = path.is_dir() if kind.endswith('_dir') else path.is_file()
        if not exists:
            raise FileNotFoundError('{} {}: {}'.format(dataset_name, kind, path))

print('Project:', PROJECT_ROOT)
print('Datasets:', ', '.join(DATASETS.keys()))

Project: C:\workspace\LSFM_pp
Datasets: 042926, 050126, 062226


In [8]:
SPOT_CACHE = {}
META_CACHE = {}

def read_spots_by_z(xml_path):
    xml_path = Path(xml_path)
    cache_key = str(xml_path.resolve())
    if cache_key in SPOT_CACHE:
        return SPOT_CACHE[cache_key]

    grouped = {}
    count = 0
    for _, element in ET.iterparse(str(xml_path), events=('end',)):
        if element.tag == 'Spot':
            z = int(round(float(element.attrib['POSITION_Z'])))
            grouped.setdefault(z, []).append((
                float(element.attrib['POSITION_X']),
                float(element.attrib['POSITION_Y']),
            ))
            count += 1
        element.clear()

    grouped = {
        z: np.asarray(points, dtype=np.float32)
        for z, points in grouped.items()
    }
    result = {'by_z': grouped, 'count': count}
    SPOT_CACHE[cache_key] = result
    return result

def read_tif_metadata(tif_path):
    cache_key = str(Path(tif_path).resolve())
    if cache_key not in META_CACHE:
        with tiff.TiffFile(cache_key) as tif_file:
            META_CACHE[cache_key] = {
                'pages': len(tif_file.pages),
                'shape': tuple(tif_file.pages[0].shape),
                'dtype': str(tif_file.pages[0].dtype),
            }
    return META_CACHE[cache_key]

@lru_cache(maxsize=12)
def read_tif_page(tif_path_string, z):
    with tiff.TiffFile(tif_path_string) as tif_file:
        return tif_file.pages[int(z)].asarray()

@lru_cache(maxsize=3)
def processed_page_paths(processed_dir_string):
    files = tuple(sorted(Path(processed_dir_string).glob('*.tif')))
    if not files:
        raise FileNotFoundError('No processed TIFF files: {}'.format(processed_dir_string))
    return files

@lru_cache(maxsize=12)
def read_processed_page(processed_dir_string, z):
    files = processed_page_paths(processed_dir_string)
    return tiff.imread(str(files[int(z)]))

def read_image_metadata(paths, image_source):
    if image_source == 'raw':
        return read_tif_metadata(paths['tif'])
    files = processed_page_paths(str(paths['processed_dir'].resolve()))
    cache_key = 'processed:' + str(paths['processed_dir'].resolve())
    if cache_key not in META_CACHE:
        with tiff.TiffFile(str(files[0])) as tif_file:
            META_CACHE[cache_key] = {
                'pages': len(files),
                'shape': tuple(tif_file.pages[0].shape),
                'dtype': str(tif_file.pages[0].dtype),
            }
    return META_CACHE[cache_key]

def read_image_page(paths, image_source, z):
    if image_source == 'raw':
        return read_tif_page(str(paths['tif'].resolve()), z)
    return read_processed_page(str(paths['processed_dir'].resolve()), z)

def clipped_view_bounds(center, full_size, zoom):
    view_size = float(full_size) / float(zoom)
    lower = float(center) - view_size / 2.0
    lower = min(max(lower, 0.0), max(float(full_size) - view_size, 0.0))
    return lower, lower + view_size

In [ ]:
dataset_widget = widgets.Dropdown(
    options=list(DATASETS.keys()), value='042926', description='Dataset:'
)
image_source_widget = widgets.Dropdown(
    options=[('Raw stack', 'raw'), ('Preprocessed slices', 'processed')],
    value='raw', description='Image:'
)
z_widget = widgets.IntSlider(
    value=59, min=0, max=179, step=1, description='Z:',
    continuous_update=False, readout=True, layout=widgets.Layout(width='75%')
)
play_widget = widgets.Play(
    value=59, min=0, max=179, step=1, interval=500, description='Play'
)
widgets.jslink((play_widget, 'value'), (z_widget, 'value'))

zoom_widget = widgets.SelectionSlider(
    options=[1, 1.5, 2, 3, 4, 6, 8, 12, 16], value=1,
    description='Zoom:', continuous_update=False, layout=widgets.Layout(width='55%')
)
center_x_widget = widgets.BoundedIntText(value=1080, min=0, max=2159, description='Center X:')
center_y_widget = widgets.BoundedIntText(value=2048, min=0, max=4095, description='Center Y:')
reset_view_button = widgets.Button(description='Reset view', icon='refresh')
center_spots_button = widgets.Button(description='Center on Spots', icon='crosshairs')

contrast_widget = widgets.FloatRangeSlider(
    value=(0.5, 99.7), min=0.0, max=100.0, step=0.1,
    description='Percentile:', continuous_update=False,
    readout_format='.1f', layout=widgets.Layout(width='75%')
)
marker_size_widget = widgets.FloatSlider(
    value=18.0, min=2.0, max=80.0, step=2.0, description='Marker:',
    continuous_update=False, layout=widgets.Layout(width='55%')
)
marker_color_widget = widgets.Dropdown(
    options=['lime', 'red', 'cyan', 'yellow', 'magenta'],
    value='lime', description='Color:'
)
figure_width_widget = widgets.FloatSlider(
    value=8.0, min=4.0, max=14.0, step=0.5, description='Figure:',
    continuous_update=False, layout=widgets.Layout(width='55%')
)
status_widget = widgets.HTML()
viewer_output = widgets.Output()
_updating_widgets = False

def current_dataset():
    return DATASETS[dataset_widget.value]

def reset_view(_=None, render=True):
    global _updating_widgets
    meta = read_image_metadata(current_dataset(), image_source_widget.value)
    height, width = meta['shape']
    _updating_widgets = True
    center_x_widget.max = width - 1
    center_y_widget.max = height - 1
    center_x_widget.value = width // 2
    center_y_widget.value = height // 2
    zoom_widget.value = 1
    _updating_widgets = False
    if render:
        render_view()

def center_on_spots(_=None):
    global _updating_widgets
    paths = current_dataset()
    spots = read_spots_by_z(paths['xml'])['by_z'].get(z_widget.value)
    if spots is None or len(spots) == 0:
        return
    _updating_widgets = True
    center_x_widget.value = int(np.clip(np.median(spots[:, 0]), 0, center_x_widget.max))
    center_y_widget.value = int(np.clip(np.median(spots[:, 1]), 0, center_y_widget.max))
    if zoom_widget.value == 1:
        zoom_widget.value = 3
    _updating_widgets = False
    render_view()

def render_view(_=None):
    if _updating_widgets:
        return
    paths = current_dataset()
    z = int(z_widget.value)
    try:
        meta = read_image_metadata(paths, image_source_widget.value)
        image = read_image_page(paths, image_source_widget.value, z)
        spot_data = read_spots_by_z(paths['xml'])
        spots = spot_data['by_z'].get(z, np.empty((0, 2), dtype=np.float32))

        low_p, high_p = contrast_widget.value
        vmin, vmax = np.percentile(image, [low_p, high_p])
        if vmax <= vmin:
            vmax = vmin + 1

        height, width = image.shape
        zoom = float(zoom_widget.value)
        x0, x1 = clipped_view_bounds(center_x_widget.value, width, zoom)
        y0, y1 = clipped_view_bounds(center_y_widget.value, height, zoom)
        visible = (
            (spots[:, 0] >= x0) & (spots[:, 0] <= x1) &
            (spots[:, 1] >= y0) & (spots[:, 1] <= y1)
        ) if len(spots) else np.zeros(0, dtype=bool)

        figure_width = float(figure_width_widget.value)
        figure_height = min(10.0, max(3.0, figure_width * (y1 - y0) / max(x1 - x0, 1)))
        fig, ax = plt.subplots(figsize=(figure_width, figure_height), dpi=120)
        ax.imshow(image, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')
        if len(spots):
            ax.scatter(
                spots[:, 0], spots[:, 1],
                s=marker_size_widget.value, facecolors='none',
                edgecolors=marker_color_widget.value, linewidths=0.8,
            )
        ax.set_xlim(x0, x1)
        ax.set_ylim(y1, y0)
        ax.set_title(
            '{} [{}]  Z={}/{} (slice {:03d})  Spots={}  visible={}  zoom={}x'.format(
                dataset_widget.value, image_source_widget.label, z, meta['pages'] - 1, z + 1, len(spots),
                int(visible.sum()), zoom_widget.value
            )
        )
        ax.set_xlabel('X (pixel)')
        ax.set_ylabel('Y (pixel)')
        fig.tight_layout()

        status_widget.value = (
            '<b>{}</b> | {} | image: {} × {} {} | total Spots: {:,} | Z {}: {:,} Spots'.format(
                dataset_widget.value, image_source_widget.label, width, height, meta['dtype'],
                spot_data['count'], z, len(spots)
            )
        )
        with viewer_output:
            clear_output(wait=True)
            display(fig)
            plt.close(fig)
    except Exception as error:
        status_widget.value = '<b style="color:red">{}</b>'.format(error)
        with viewer_output:
            clear_output(wait=True)
            print(type(error).__name__ + ':', error)

def change_dataset(change):
    global _updating_widgets
    if change.get('name') != 'value':
        return
    paths = current_dataset()
    meta = read_image_metadata(paths, image_source_widget.value)
    _updating_widgets = True
    z_widget.max = meta['pages'] - 1
    play_widget.max = meta['pages'] - 1
    z_widget.value = min(z_widget.value, z_widget.max)
    _updating_widgets = False
    reset_view(render=False)
    status_widget.value = '<i>Loading and indexing {} Spots...</i>'.format(dataset_widget.value)
    render_view()

def change_image_source(change):
    if change.get('name') != 'value':
        return
    reset_view(render=False)
    render_view()

reset_view_button.on_click(reset_view)
center_spots_button.on_click(center_on_spots)
dataset_widget.observe(change_dataset, names='value')
image_source_widget.observe(change_image_source, names='value')
for widget in [
    z_widget, zoom_widget, center_x_widget, center_y_widget, contrast_widget,
    marker_size_widget, marker_color_widget, figure_width_widget,
]:
    widget.observe(render_view, names='value')

controls = widgets.VBox([
    widgets.HBox([dataset_widget, image_source_widget]),
    status_widget,
    widgets.HBox([play_widget, z_widget]),
    widgets.HBox([zoom_widget, center_x_widget, center_y_widget]),
    widgets.HBox([reset_view_button, center_spots_button]),
    contrast_widget,
    widgets.HBox([marker_size_widget, marker_color_widget, figure_width_widget]),
])

reset_view(render=False)
display(controls, viewer_output)
render_view()

Output()